In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal.windows import tukey
from pathlib import Path
from scipy.signal import welch

In [144]:
def window_organ_signal(signal, sample_rate, start_sec, end_sec, fade_fraction=0.1):

    """
    Ritaglia il segnale e applica una finestra di Tukey per evitare click e leakage.
    
    signal: Array numpy del segnale audio.
    sample_rate: Frequenza di campionamento.
    start_sec: Secondo di inizio del taglio.
    end_sec: Secondo di fine del taglio.
    fade_fraction: Percentuale della finestra dedicata al fade (0-1).
    return: Segnale ritagliato e finestrato.
    """

    # 1. Ritaglio temporale
    start_sample = int(start_sec * sample_rate)
    end_sample = int(end_sec * sample_rate)
    trimmed_signal = signal[start_sample:end_sample]
    
    # 2. Creazione della finestra di Tukey
    # fade_fraction definisce quanto i bordi sono smussati.
    # Se 0 è una rettangolare, se 1 è una finestra di Hann.
    win = tukey(len(trimmed_signal), alpha = fade_fraction)
    
    # 3. Applicazione
    windowed_signal = trimmed_signal * win
    
    return windowed_signal

In [145]:
def process_organ_notes(folder_path, start_sec=0.5, end_sec=2.5):
    
    data_dir = Path(folder_path)
    files = sorted(data_dir.glob('note_*.wav'))
    results = {}

    for file_path in files:
        samplerate, data = wavfile.read(file_path)
        
        # 1. Normalizzazione e gestione Stereo -> Mono
        if data.dtype == np.int16:
            data = data / 32768.0
            
        if len(data.shape) > 1:
            signal = np.mean(data, axis=1)
        else:
            signal = data

        # 2. CHIAMATA ALLA FUNZIONE ESTERNA
        # Usiamo la funzione che abbiamo scritto prima!
        final_signal = window_organ_signal(signal, samplerate, start_sec, end_sec)
        
        # 3. FFT e Analisi
        n = len(final_signal)
        fft_values = np.fft.fft(final_signal)
        frequencies = np.fft.fftfreq(n, d=1/samplerate)[:n//2]
        power_spectrum = np.abs(fft_values[:n//2])**2
        
        results[file_path.name] = {
            'freq': frequencies,
            'psd': power_spectrum,
            'rate': samplerate
        }
        print(f"Elaborato: {file_path.name}")

    return results

In [146]:
"""Il Problema: La FFT \"Grezza\" (Periodogramma)
Se prendi un segnale intero (come hai fatto all'inizio del tuo notebook) e gli applichi una singola, gigantesca Trasformata di Fourier (FFT), ottieni un risultato molto instabile. Se c'è anche un minimo rumore di fondo, il grafico esploderà in una miriade di picchi \"seghettati\" e frastagliati. Questo fenomeno si chiama alta varianza: il risultato varia tantissimo a causa del rumore casuale, nascondendo le vere frequenze del segnale.

La Soluzione: Il Metodo di Welch
Il metodo di Welch risolve questo problema usando la statistica, in particolare la forza della media matematica. Lo fa in 4 passaggi fondamentali:

1.⁠ ⁠Segmentazione (Divisione in blocchi)
Invece di fare una singola FFT lunga quanto tutto il file audio, Welch prende il segnale e lo \"affetta\" in tanti blocchi più corti. La lunghezza di questi blocchi è il famoso parametro nperseg (Number of Points PER SEGment).

2.⁠ ⁠Sovrapposizione (Overlap)
Se tagliassimo il segnale come se fossero mattoni, perderemmo delle informazioni proprio nei punti di taglio. Per evitare questo, i segmenti vengono fatti sovrapporre (di solito del 50%). Quindi il primo segmento va da 0 a 1000, il secondo va da 500 a 1500, il terzo da 1000 a 2000, e così via.

3.⁠ ⁠Finestratura (Windowing)
A ciascuno di questi segmenti viene applicata una \"finestra\" matematica (spesso chiamata finestra di Hann o Hamming). Questa curva smussa i bordi di ogni singolo segmento portandoli dolcemente a zero. Questo previene un brutto difetto matematico chiamato Spectral Leakage (dispersione spettrale), che altrimenti creerebbe frequenze fantasma che non esistono davvero.

4.⁠ ⁠FFT e Media
A questo punto, il metodo calcola la potenza della FFT per ogni singolo segmento. Se hai diviso il file in 100 pezzetti, avrai 100 grafici FFT separati. Infine, fa la media di tutti questi 100 grafici.

Il risultato e il \"Compromesso di Welch\"
Facendo la media, il rumore casuale di fondo si annulla a vicenda (perché a volte va su, a volte va giù), mentre le frequenze stabili (come la nota del tuo organo) si sommano e si confermano. Il risultato è quel grafico bellissimo e smussato che hai ottenuto.

C'è però un compromesso fondamentale (Trade-off) che devi decidere quando scegli il parametro nperseg:

nperseg PICCOLO (es. 256): Avrai tantissimi segmenti di cui fare la media. Il grafico sarà incredibilmente liscio e senza rumore, ma i picchi saranno molto larghi e \"cicciotti\" (poca risoluzione in frequenza). Farai fatica a distinguere due frequenze molto vicine.

nperseg GRANDE (es. 4096 o 8192): Avrai pochi segmenti di cui fare la media. I picchi saranno sottilissimi e precisi (altissima risoluzione in frequenza), ma la linea di base tornerà ad essere un po' frastagliata e rumorosa."""

def analizza_nota_welch_mono(file_path, start_sec=0.5, end_sec=1.5, nperseg=8192):
    """
    Converte in mono, ritaglia e calcola lo spettro mediato.
    """
    # 1. Caricamento e Normalizzazione
    samplerate, data = wavfile.read(file_path)
    if data.dtype == np.int16:
        data = data / 32768.0

    # 2. Conversione in MONO (Media dei canali)
    if len(data.shape) > 1:
        signal_mono = np.mean(data, axis=1)
    else:
        signal_mono = data

    # 3. Ritaglio e Windowing (Sostegno)
    # Usiamo la tua funzione window_organ_signal
    signal_cut = window_organ_signal(signal_mono, samplerate, start_sec, end_sec, fade_fraction=0.05)

    # 4. Calcolo PSD con Welch
    # Ho alzato nperseg a 8192 per darti più risoluzione sulle note basse
    freqs, psd = welch(signal_cut, fs=samplerate, nperseg=nperseg)

    """# 5. Plotting (Singolo grafico, molto più chiaro)
    plt.figure(figsize=(10, 5))
    plt.plot(freqs, 10 * np.log10(psd + 1e-12), color='forestgreen')
    plt.title(f"Spettro di Potenza (Mono) - {Path(file_path).name}")
    plt.xlabel('Frequenza (Hz)')
    plt.ylabel('Potenza (dB/Hz)')
    plt.xlim(0, 5000)
    
    # Griglia più fitta in x: major ogni 100 Hz, minor ogni 20 Hz
    ax = plt.gca()
    ax.set_xticks(np.arange(0, 5001, 300))
    #ax.set_xticks(np.arange(0, 5001, 20), minor=True)
    ax.grid(True, which='major', alpha=0.35)
    #ax.grid(True, which='minor', axis='x', alpha=0.2, linestyle=':')

    plt.tight_layout()
    plt.show()"""
    
    return freqs, psd

In [149]:
def mcd_euclide_float(a, b, tolleranza=2.0):
    """
    Versione dell'algoritmo di Euclide per numeri float.
    Si ferma quando il resto è inferiore alla tolleranza.
    """
    while b > tolleranza:
        a, b = b, a % b
    return a

def identifica_f0_euclide(freqs, psd, num_picchi=5):
    """
    Identifica la fondamentale calcolando il Massimo Comun Divisore 
    tra le frequenze dei picchi principali usando il metodo di Euclide.
    """
    # 1. Trova i picchi principali (semplice ricerca locale)
    soglia = np.max(psd) * 0.1
    picchi_idx = []
    for i in range(1, len(psd) - 1):
        if psd[i] > psd[i-1] and psd[i] > psd[i+1] and psd[i] > soglia:
            picchi_idx.append(i)
    
    f_picchi = freqs[picchi_idx[:num_picchi]]
    
    if len(f_picchi) < 2:
        return f_picchi[0] if len(f_picchi) == 1 else 0

    # 2. Applichiamo Euclide in cascata sui picchi trovati
    # Calcoliamo l'MCD tra il primo e il secondo, poi il risultato col terzo, ecc.
    # Usiamo una tolleranza legata alla risoluzione della FFT (es. 2-5 Hz)
    res_mcd = f_picchi[0]
    for i in range(1, len(f_picchi)):
        res_mcd = mcd_euclide_float(res_mcd, f_picchi[i], tolleranza=2.5)
    
    # 3. Controllo di plausibilità
    # A volte l'MCD può risultare in una sottarmonica (es. f0/2). 
    # Se il valore è troppo basso (es. < 15Hz per un organo), potrebbe esserci un errore.
    
    return res_mcd

def calcola_ampiezza_armoniche(freqs, psd, f0, num_overtones=5, search_width_hz=10.0):
    """
    Stima l'ampiezza delle armoniche cercando il massimo della PSD
    in una finestra attorno a ciascun multiplo di f0.
    """
    armoniche = {}
    if f0 <= 0:
        return armoniche

    for overtone in range(2, num_overtones + 2):
        target_freq = overtone * f0
        mask = np.abs(freqs - target_freq) <= search_width_hz
        if np.any(mask):
            local_freqs = freqs[mask]
            local_psd = psd[mask]
            peak_idx = np.argmax(local_psd)
            armoniche[f"{overtone}x"] = {
                "target_freq": float(target_freq),
                "peak_freq": float(local_freqs[peak_idx]),
                "amplitude": float(local_psd[peak_idx])
            }
        else:
            armoniche[f"{overtone}x"] = {
                "target_freq": float(target_freq),
                "peak_freq": None,
                "amplitude": None
            }

    return armoniche


freqs, psd = analizza_nota_welch_mono('data/note_91.wav')
f0 = identifica_f0_euclide(freqs, psd)
overtones = calcola_ampiezza_armoniche(freqs, psd, f0)
print(f"La frequenza fondamentale stimata è: {f0:.2f} Hz")
print(overtones)

La frequenza fondamentale stimata è: 662.11 Hz
{'2x': {'target_freq': 1324.21875, 'peak_freq': 1324.21875, 'amplitude': 18111606465549.99}, '3x': {'target_freq': 1986.328125, 'peak_freq': 1986.328125, 'amplitude': 2015127397572.1907}, '4x': {'target_freq': 2648.4375, 'peak_freq': 2642.578125, 'amplitude': 1210989872256.8623}, '5x': {'target_freq': 3310.546875, 'peak_freq': 3304.6875, 'amplitude': 9875466691.6578}, '6x': {'target_freq': 3972.65625, 'peak_freq': 3966.796875, 'amplitude': 100884049219.59607}}


/var/folders/xt/7jwm3jpn57sgpkrqpy1g1bph0000gn/T/ipykernel_27841/1856061491.py:33: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, data = wavfile.read(file_path)


In [ ]:
rows = []

for i in range(280):
    note_name = f"note_{i+1}.wav"
    freqs, psd = analizza_nota_welch_mono(
        f"data/{note_name}",
        start_sec=0.5,
        end_sec=2.5,
        nperseg=8192
    )
    f0 = identifica_f0_euclide(freqs, psd)
    overtones = calcola_ampiezza_armoniche(freqs, psd, f0)

    row = {
        "note": note_name,
        "f0": f0,
    }
    for overtone_label, overtone_data in overtones.items():
        row[f"{overtone_label}_target_freq"] = overtone_data["target_freq"]
        row[f"{overtone_label}_peak_freq"] = overtone_data["peak_freq"]
        row[f"{overtone_label}_amplitude"] = overtone_data["amplitude"]
    rows.append(row)

results_df = pd.DataFrame(rows)
print(results_df)

/var/folders/xt/7jwm3jpn57sgpkrqpy1g1bph0000gn/T/ipykernel_27841/1856061491.py:33: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, data = wavfile.read(file_path)


{'note_1.wav': {'f0': np.float64(1564.453125), 'overtones': {'2x': {'target_freq': 3128.90625, 'peak_freq': 3128.90625, 'amplitude': 13511398768769.93}, '3x': {'target_freq': 4693.359375, 'peak_freq': 4687.5, 'amplitude': 81023593376.75816}, '4x': {'target_freq': 6257.8125, 'peak_freq': 6251.953125, 'amplitude': 1837226711.6592188}, '5x': {'target_freq': 7822.265625, 'peak_freq': 7816.40625, 'amplitude': 280335444.66912824}, '6x': {'target_freq': 9386.71875, 'peak_freq': 9380.859375, 'amplitude': 43900821434.660995}}}, 'note_2.wav': {'f0': np.float64(1482.421875), 'overtones': {'2x': {'target_freq': 2964.84375, 'peak_freq': 2964.84375, 'amplitude': 38827388451.759224}, '3x': {'target_freq': 4447.265625, 'peak_freq': 4447.265625, 'amplitude': 7420148011.177807}, '4x': {'target_freq': 5929.6875, 'peak_freq': 5929.6875, 'amplitude': 108918662648.5295}, '5x': {'target_freq': 7412.109375, 'peak_freq': 7412.109375, 'amplitude': 48552037749.04522}, '6x': {'target_freq': 8894.53125, 'peak_freq